In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint
from IPython.display import Image, Markdown, display

from data.share_data import load_share_data
from scripts.madrl import SCHEMES, load_madrl_record, plot_madrl_outputs, run_madrl_scheme_experiment
from utils.run_artifacts import load_experiment_context


In [ ]:
run_dir = Path("artifacts/runs/20260504_234339_01e73bd1")
scheme_name = "madrl_base"
# Set True only when you intentionally want 500-episode retraining.
retrain = False

cfg, run_dir = load_experiment_context(run_dir)
share_data = load_share_data(run_dir / "share_data", cfg)
assert int(share_data.manifest["sequence_length"]) == int(cfg.obs.sequence_length)
spec = next(item for item in SCHEMES if item["scheme"] == scheme_name)
train_episodes = 500
w_voltage, w_line, w_trafo = spec["reward"]

display(Markdown("# MADRL + No Safety + LSTM Forecast"))
display(pd.DataFrame([{
    "run_dir": str(run_dir),
    "scheme": scheme_name,
    "controller": spec["controller"],
    "train_episodes": train_episodes,
    "retrain": bool(retrain),
    "device": cfg.runtime.device,
    "train_window_days": int(cfg.env.train_window_days),
    "train_episode_steps": int(cfg.env.episode_steps) * int(cfg.env.train_window_days),
    "action_dim": int(cfg.model.action_dim),
    "max_charge_rate": float(cfg.env.max_charge_rate),
    "init_soc": float(cfg.env.init_soc),
    "train_init_soc_low": float(cfg.env.train_init_soc_low),
    "train_init_soc_high": float(cfg.env.train_init_soc_high),
    "batch_size": int(cfg.train.batch_size),
    "gamma": float(cfg.algo.gamma),
    "tau": float(cfg.algo.tau),
    "hidden_dim": int(cfg.model.hidden_dim),
    "n_step_return": int(cfg.train.n_step_return),
    "w_voltage_pen": float(w_voltage),
    "w_line_pen": float(w_line),
    "w_trafo_pen": float(w_trafo),
}]))


In [ ]:
record_manifest = run_dir / "results" / "lstm" / spec["controller"] / "record" / "manifest.json"
reward_curve_path = run_dir / "tables" / f"madrl_reward_curves_{scheme_name}.csv"
train_summary_path = run_dir / "tables" / f"madrl_train_summary_{scheme_name}.csv"

if retrain:
    result = run_madrl_scheme_experiment(cfg, run_dir, share_data, spec, episodes=train_episodes)
else:
    missing = [path for path in (record_manifest, reward_curve_path, train_summary_path) if not path.exists()]
    if missing:
        raise FileNotFoundError("Cached MADRL result is incomplete. Set retrain=True and rerun this cell. Missing: " + ", ".join(str(path) for path in missing))
    result = {
        "records": {scheme_name: load_madrl_record(run_dir, spec["controller"])},
        "train_summary": pd.read_csv(train_summary_path),
        "reward_curves": pd.read_csv(reward_curve_path),
    }

display(Markdown("## Train Summary"))
display(result["train_summary"])

saved = result["records"][scheme_name]
display(Markdown("## Test Metrics"))
display(saved["metrics_df"])
display(Markdown("## Agent Summary"))
display(saved["rollout"].summary)
pprint({"record_dir": str(saved["record_dir"])})


In [ ]:
plt.close("all")
figures = plot_madrl_outputs(result, run_dir)
_plot_specs = [
    ("1. 电力电量平衡", "madrl_base_power_balance"),
    ("2. 电价", "madrl_base_price"),
    ("3. 总体储能充放功率与 SoC", "madrl_base_battery_soc"),
    ("4. 电压", "madrl_base_voltage"),
    ("5. 净负荷", "madrl_base_net_load"),
    ("6. 训练奖励", "madrl_base_learning_curve"),
]
for title, name in _plot_specs:
    print(title)
    display(Image(filename=str(run_dir / "figures" / f"{name}.png")))
    plt.close(figures[name])
